# Graph Neural Net

datacamp tutorial: https://www.datacamp.com/tutorial/comprehensive-introduction-graph-neural-networks-gnns-tutorial?dc_referrer=https%3A%2F%2Fwww.google.com%2F

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd

In [ ]:
# df = pd.read_csv('../data/raw/credit_data.csv')
# data = df.values[0]
# print(data)

Example using dataset from website (uses Planetoid() + NormaliseFeatures() so it's not a standard dataframe like in our other models)

In [ ]:
from torch_geometric.datasets import Planetoid
from torch_geometric.transforms import NormalizeFeatures

dataset = Planetoid(root='data/Planetoid', name='Cora', transform=NormalizeFeatures())

print(f'Dataset: {dataset}:')
print('======================')
print(f'Number of graphs: {len(dataset)}')
print(f'Number of features: {dataset.num_features}')
print(f'Number of classes: {dataset.num_classes}')

data = dataset[0]  # Get the first graph object.
print(data)

In [ ]:
# df.head()

In [ ]:
# print(df.shape)

### CGN Model (Graph Convolutional Network)

In [ ]:
from torch_geometric.nn import GCNConv

In [ ]:
class GCN(nn.Module):
    def __init__(self, hidden_channels):
        super().__init__()
        torch.manual_seed(1234567)
        self.conv1 = GCNConv(dataset.num_features, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, dataset.num_classes)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = x.relu()
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.conv2(x, edge_index)
        return x

model = GCN(hidden_channels=16)
print(model)


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

def visualize(h, color):
    z = TSNE(n_components=2).fit_transform(h.detach().cpu().numpy())

    plt.figure(figsize=(10,10))
    plt.xticks([])
    plt.yticks([])

    plt.scatter(z[:, 0], z[:, 1], s=70, c=color, cmap="Set2")
    plt.show()

In [ ]:
model.eval()

# visualize raw, untrained data
out = model(data.x, data.edge_index)
visualize(out, color=data.y)

In [ ]:
# set up train and test functions

model = GCN(hidden_channels=16)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
criterion = nn.CrossEntropyLoss()

def train():
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index)
    loss = criterion(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    return loss

def test():
    model.eval()
    out = model(data.x, data.edge_index)
    pred = out.argmax(dim=1)
    test_correct = pred[data.test_mask] == data.y[data.test_mask]
    test_acc = int(test_correct.sum()) / int(data.test_mask.sum())
    return test_acc

In [ ]:
# train model

for epoch in range(1, 101):
    loss = train()

    if epoch % 10 == 0:
        print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}')

In [ ]:
# calculate accuracy

test_acc = test()
print(f'Test Accuracy: {test_acc:.4f}')

In [ ]:
# visualize trained model

model.eval()
out = model(data.x, data.edge_index)
visualize(out, color=data.y)

### GATConv Model (Graph Attention Network)

In [ ]:
from torch_geometric.nn import GATConv

In [ ]:
class GAT(nn.Module):
    def __init__(self, hidden_channels, heads):
        super().__init__()
        torch.manual_seed(1234567)
        self.conv1 = GATConv(dataset.num_features, hidden_channels, heads=heads)
        self.conv2 = GATConv(hidden_channels * heads, dataset.num_classes, heads=heads)

    def forward(self, x, edge_index):
        x = F.dropout(x, p=0.6, training=self.training)
        x = self.conv1(x, edge_index)
        x = F.elu(x)
        x = F.dropout(x, p=0.6, training=self.training)
        x = self.conv2(x, edge_index)
        return x


In [ ]:
model = GAT(hidden_channels=8, heads=8)
print(model)

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)
criterion = nn.CrossEntropyLoss()

In [ ]:
# same train and test functions as GCN, do I have to declare them again??

def train():
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index)
    loss = criterion(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    return loss

def test(mask):
    model.eval()
    out = model(data.x, data.edge_index)
    pred = out.argmax(dim=1)
    correct = pred[mask] == data.y[mask]
    acc = int(correct.sum()) / int(mask.sum())
    return acc

In [ ]:
val_acc_all = []
test_acc_all = []

In [ ]:
for epoch in range(1, 101):
    loss = train()
    val_acc = test(data.val_mask)
    test_acc = test(data.test_mask)
    val_acc_all.append(val_acc)
    test_acc_all.append(test_acc)

    if epoch % 10 == 0:
        print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, Val Acc: {val_acc:.4f}, Test Acc: {test_acc:.4f}')

In [ ]:
import numpy as np

In [ ]:
plt.figure(figsize=(12,8))
plt.plot(np.arange(1, len(val_acc_all) + 1), val_acc_all, label='Validation accuracy', c='blue')
plt.plot(np.arange(1, len(test_acc_all) + 1), test_acc_all, label='Testing accuracy', c='red')
plt.xlabel('Epochs')
plt.ylabel('Accurarcy')
plt.title('GATConv')
plt.legend(loc='lower right', fontsize='x-large')
plt.show()

In [ ]:
model.eval()

out = model(data.x, data.edge_index)
visualize(out, color=data.y)

### Next Steps:  
Converting tabular data to graph data for GNN: https://youtu.be/AQU3akndun4?si=g8V4zo36SQNdGLW-   
Fake News detection using GNN: https://youtu.be/QAIVFr24FrA?si=cw2sDav_jZJMuaBD   
Self-/Unsupervised GNN training: https://youtu.be/3XTuhchTWd8?si=Jxi8Xn9hPOD6EQQF 